# GeoAPI workflow examples

This notebook shows example automated workflows using the current `idrogeno` endpoints.

Current endpoints used here:

- `GET /pozzi/`
- `GET /pozzi/{id_pozzo}/`
- `GET /pozzi/{id_pozzo}/stratigrafia/`
- `POST /pozzi/nearest/`
- `POST /pozzi/in-geometry/`


## 1. Setup

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://geoapi.rse-web.it/api/idrogeno"

def get_json(response):
    """Raise a useful error if the API call failed, otherwise return JSON."""
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        print("Status code:", response.status_code)
        print("Response text:", response.text)
        raise exc
    return response.json()


## 2. Example: get wells (pozzi) filtered by regione/provincia

This uses the `GET /pozzi/` endpoint with optional query parameters.

In [2]:
params = {
    "regione": "Lombardia",
    # "provincia": "Milano",
}

response = requests.get(f"{BASE_URL}/pozzi/", params=params)
pozzi = get_json(response)

df_pozzi = pd.DataFrame(pozzi)
df_pozzi.head()


,id_pozzo,nome_pozzo,ub,regione,provincia,x_32632,y_32632,data,geom_wkt
0,44,AGNADELLO 001,T,Lombardia,CR,540015.12632,5.032804e+06,None,POINT (540015.12632 5032804.1702)
1,142,ALFA,T,Lombardia,PV,0.00000,0.000000e+00,None,POINT (0 0)
2,144,ALFIANELLO 001,T,Lombardia,BS,589077.49143,5.015591e+06,None,POINT (589077.49143 5015591.13489)
3,145,ALFIANELLO 002,T,Lombardia,BS,590040.82174,5.015265e+06,None,POINT (590040.82174 5015265.26602)
4,399,ANTEGNATE 001,T,Lombardia,BG,562954.08214,5.037035e+06,None,POINT (562954.08214 5037035.33174)


## 3. Example: find the nearest `n` wells (pozzi) to a point

This uses `POST /pozzi/nearest/`.

The point is sent as GeoJSON. The `srid` tells the API which coordinate reference system the input geometry uses.

In [3]:
nearest_payload = {
    "srid": 32632,
    "geometry": {
        "type": "Point",
        "coordinates": [500000, 5050000],
    },
    "n": 3,
}

response = requests.post(f"{BASE_URL}/pozzi/nearest/", json=nearest_payload)
nearest = get_json(response)

nearest["count"], nearest["results"][:1]


(3,
 [{'id_pozzo': '2122',
   'nome_pozzo': 'CESATE 001',
   'ub': 'T',
   'regione': 'Lombardia',
   'provincia': 'MI',
   'x_32632': 505701.8991,
   'y_32632': 5047831.801,
   'data': None,
   'geom_wkt': 'POINT (505701.8991 5047831.801)',
   'distance_m': 6100.22460653554}])

In [4]:
df_nearest = pd.DataFrame(nearest["results"])
df_nearest[["id_pozzo", "nome_pozzo", "regione", "provincia", "distance_m"]].head()


,id_pozzo,nome_pozzo,regione,provincia,distance_m
0,2122,CESATE 001,Lombardia,MI,6100.224607
1,457,ARESE 001,Lombardia,MI,7375.317982
2,2903,CUSANO MILANINO 002,Lombardia,MI,11562.584970


## 4. Example: take the nearest well (pozzo) and get its stratigraphy (stratigrafia)

This chains two endpoints:

1. `POST /pozzi/nearest/`
2. `GET /pozzi/{id_pozzo}/stratigrafia/`

In [5]:
if nearest["count"] == 0:
    raise ValueError("No pozzi found near the input point.")

nearest_pozzo = nearest["results"][0]
id_pozzo = nearest_pozzo["id_pozzo"]

response = requests.get(f"{BASE_URL}/pozzi/{id_pozzo}/stratigrafia/")
stratigrafia_response = get_json(response)

stratigrafia_response


{'id_pozzo': 2122,
 'count': 6,
 'results': [{'id': 2864,
   'id_pozzo': 2122,
   'profondita_min': 400,
   'profondita_max': 520,
   'eta': 'Quaternario',
   'formazione': None,
   'litologia': 'sabbia argillosa',
   'note_litologia': 'sabbie e sabbie argillose con intercalazioni di argilla e qualche ciottolo.',
   'pendenza': None},
  {'id': 2865,
   'id_pozzo': 2122,
   'profondita_min': 520,
   'profondita_max': 669,
   'eta': 'Pliocene medio',
   'formazione': None,
   'litologia': 'sabbia argillosa',
   'note_litologia': 'sabbie e sabbie argillose con intercalazioni di argilla e qualche ciottolo.',
   'pendenza': None},
  {'id': 2866,
   'id_pozzo': 2122,
   'profondita_min': 669,
   'profondita_max': 730,
   'eta': 'Pliocene medio',
   'formazione': None,
   'litologia': 'sabbia argillosa',
   'note_litologia': 'sabbie argillose e argille sabbiose con intercalazioni di sabbia ed argilla',
   'pendenza': None},
  {'id': 2867,
   'id_pozzo': 2122,
   'profondita_min': 730,
   'pro

In [6]:
results = stratigrafia_response.get("results", stratigrafia_response)
df_stratigrafia = pd.DataFrame(results)
df_stratigrafia.head()


,id,id_pozzo,profondita_min,profondita_max,eta,formazione,litologia,note_litologia,pendenza
0,2864,2122,400,520,Quaternario,None,sabbia argillosa,sabbie e sabbie argillose con intercalazioni d...,None
1,2865,2122,520,669,Pliocene medio,None,sabbia argillosa,sabbie e sabbie argillose con intercalazioni d...,None
2,2866,2122,669,730,Pliocene medio,None,sabbia argillosa,sabbie argillose e argille sabbiose con interc...,None
3,2867,2122,730,869,Pliocene inferiore,None,sabbia argillosa,sabbie argillose e argille sabbiose con interc...,None
4,2868,2122,869,879,Pliocene inferiore,None,argilla,argille,None


## 5. Example: get all wells (pozzi) inside a polygon

This uses `POST /pozzi/in-geometry/`.


In [7]:
polygon_payload = {
    "srid": 32632,
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [450000, 5000000],
                [650000, 5000000],
                [650000, 5200000],
                [450000, 5200000],
                [450000, 5000000],
            ]
        ],
    },
}

response = requests.post(f"{BASE_URL}/pozzi/in-geometry/", json=polygon_payload)
inside = get_json(response)

inside["count"]


711

In [8]:
df_inside = pd.DataFrame(inside["results"])
df_inside.head()


,id_pozzo,nome_pozzo,ub,regione,provincia,x_32632,y_32632,data,geom_wkt
0,44,AGNADELLO 001,T,Lombardia,CR,540015.12632,5.032804e+06,None,POINT (540015.12632 5032804.1702)
1,144,ALFIANELLO 001,T,Lombardia,BS,589077.49143,5.015591e+06,None,POINT (589077.49143 5015591.13489)
2,145,ALFIANELLO 002,T,Lombardia,BS,590040.82174,5.015265e+06,None,POINT (590040.82174 5015265.26602)
3,399,ANTEGNATE 001,T,Lombardia,BG,562954.08214,5.037035e+06,None,POINT (562954.08214 5037035.33174)
4,452,ARCONATE 001,T,Lombardia,MI,487593.32074,5.040864e+06,None,POINT (487593.32074 5040863.6456)
